[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/qem/qem.ipynb)

# Quadric Error Metrics

This example simplifies a triangulated surface by collapsing edges into vertices placed where they disturb the surrounding planes least. Every triangle lies in a supporting plane, and moving a vertex off the planes of its incident triangles incurs the sum of its squared perpendicular distances to them. Collapsing an edge merges its two endpoints into one replacement vertex serving both triangle stars, so on a sharp ridge the replacement stays along the ridge and shifts toward the steeper corner.

In 3D projective geometric algebra, a plane is modelled as a vector and a point is modelled as an antivector, so their regressive pairing `plane & point` measures the signed distance from a unit plane. A plane's dyad `plane * (plane & Point)` maps points to polar planes whose pairing back with the point evaluates the squared distance, and summing incident plane dyads onto vertices and edges makes edge contraction a sum of maps. Solving `quadric(Direction)` against `-quadric(midpoint)` on spatial directions keeps unit homogeneous weight while choosing the minimizer nearest the midpoint along flat patches or creases.

In the homogeneous matrix notation of [Garland and Heckbert](https://www.cs.cmu.edu/~garland/quadrics/), each unit plane is packed into a four-vector $\mathbf p = (n_x, n_y, n_z, d)^{\mathsf T}$ with $d = -\mathbf n \cdot \mathbf x_0$, its error quadric is accumulated as the $4 \times 4$ outer product $Q = \sum \mathbf p\mathbf p^{\mathsf T}$, and minimizing $\tilde{\mathbf v}^{\mathsf T} Q\,\tilde{\mathbf v}$ at $\tilde{\mathbf v} = (x, y, z, 1)^{\mathsf T}$ slices out the upper-left $3 \times 3$ block and top-right $3 \times 1$ column of $Q$ with a pseudoinverse fallback when that block is singular.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext
from numga.algebras import PGA3D as ga, VGA3D
from numga.topology import Topology
from examples import MODEL_DIR
from examples.util import shapes
from examples.util.animation import save_figure, show_states
from examples.surfaces.qem import plumbing, render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
# A quadric maps each point to its polar plane.
Quadric = ga.gatype((Plane, Point))
Spatial = Plane.derive.within(VGA3D)                # Vector+
Direction = Spatial.derive.dual                     # Antivector-

## Supporting planes on a ridge patch

A triangulated ridge patch runs from a steep corner apex to a gentler crease continuation, flanked by two shared ridge triangles with steeper triangles around the corner and shallower ones at the far end. Joining each triangle's three corners and normalizing yields its unit supporting plane.

In [ ]:
# A triangulated patch with a sharp ridge from corner `a` to gentler end `b`, listing `a` and `b` first and the two flanking triangles first.
ridge_vertices = [
    [ 0.35,  0.0,   0.25],  # 0: vertex a (sharp corner apex)
    [-0.35,  0.0,   0.25],  # 1: vertex b (crease continuation)
    [ 0.0,   0.55, -0.15],  # 2: left base vertex
    [ 0.0,  -0.55, -0.15],  # 3: right base vertex
    [ 0.75,  0.0,  -0.15],  # 4: front tip (steep corner at a)
    [-0.85,  0.0,  -0.15],  # 5: back tip (gentle slope at b)
]
ridge_faces = np.array([
    [0, 1, 2],  # 0: left flank (shared by edge a-b)
    [1, 0, 3],  # 1: right flank (shared by edge a-b)
    [0, 2, 4],  # 2: corner front-left (incident to a only)
    [0, 4, 3],  # 3: corner front-right (incident to a only)
    [1, 5, 2],  # 4: slope back-left (incident to b only)
    [1, 3, 5],  # 5: slope back-right (incident to b only)
])
ridge_edge = np.array([0, 1])
incident_a = np.array([0, 1, 2, 3])
incident_b = np.array([0, 1, 4, 5])
surviving_faces = np.array([2, 3, 4, 5])

In [ ]:
ridge = Topology(len(ridge_vertices), ridge_faces)
vertices = mv.point(ridge_vertices).field()                                     # Point-[V]
corners = ridge.incidence[2, 0] * vertices                                      # [3] Point-[F]
face_planes = (corners[0] & corners[1] & corners[2]).normalized()               # Plane-[F]

## Collapsing a ridge edge

Each vertex's error quadric sums the plane dyads of its incident triangles, and contracting the ridge edge adds the quadrics of its two endpoints. At the optimal replacement vertex, the spatial part of the polar plane vanishes while the two flanking triangles degenerate along the collapsed edge.

In [ ]:
def collapse_position(quadric: Quadric, midpoint: Point) -> Point:
    """The unit-weight minimum of `quadric` nearest `midpoint` resolves unconstrained flat or crease directions (`[...] Point-`)."""
    # Displacing by `Direction` keeps unit homogeneous weight; zeroing the spatial polar plane leaves the minimum error in `w`.
    gradient = quadric(Direction).cast(Spatial)                                 # [...] Plane+ <- Direction-
    residual = quadric(midpoint).cast(Spatial)                                  # [...] Plane+
    correction = gradient.lstsq(-residual)                                      # [...] Direction-
    return midpoint + correction                                                # [...] Point-


# Sum each triangle's squared-distance plane dyad onto its incident vertices.
dyads = face_planes * (face_planes & Point)                                     # (Plane+ <- Point-)[F]
quadrics = ridge.incidence[2, 0].sum(axis=0).adjugate() * dyads                 # (Plane+ <- Point-)[V]
ends = ridge.reading(ridge_edge, 0)                                             # Scalar+[2 <- V]
quadric_a, quadric_b = (ends * quadrics).batch()                                # [] Plane+ <- Point-, Plane+ <- Point-
# Collapsing the edge adds the endpoint quadrics and places the replacement vertex at their unit-weight minimum.
edge_quadric = quadric_a + quadric_b                                            # [] Plane+ <- Point-
optimum = collapse_position(edge_quadric, (ends * vertices).batch().mean(axis=0))   # [] Point-
collapsed_vertices = vertices + ends.adjugate() * (optimum - ends * vertices)   # Point-[V]
save_figure(render.draw_qem(ridge, vertices, incident_a, incident_b, quadric_a, quadric_b, edge_quadric, optimum,
                            collapsed_vertices, surviving_faces), "qem_mesh_simplification");

The left and middle panels show each endpoint's incident triangle star and translucent error level-set ellipsoid (compact in rose around the three transverse directions at the sharp corner, elongated in violet along the crease at the gentler end), while the right panel shows the contracted patch around the green optimal vertex over the faint initial patch.

In [ ]:
endpoints = (ends * vertices).batch()                                           # [2] Point-
print("the best vertex:\n", optimum.coefficient_table())
print("its error, against the endpoints':", (edge_quadric(optimum) & optimum).to_array(), (edge_quadric(endpoints) & endpoints).to_array())

## Progressive surface simplification

Applying the same contraction step repeatedly coarsens the closed Stanford bunny across descending triangle-count targets. Every merged vertex retains the sum of its initial predecessors' quadrics so error continues to measure deviation from the original surface, while each pass contracts disjoint vertex stars that preserve face orientations and manifold connectivity.

In [ ]:
def simplify(topology: Topology, vertices: Point, targets: np.ndarray) -> Iterator[tuple[Topology, Point]]:
    """Yields a closed triangulated surface and its quadric-optimal contractions at descending triangle-count `targets`.

    Merged vertices accumulate the quadrics of their initial predecessors so errors remain measured against the initial face planes, while each pass contracts disjoint neighbourhoods that preserve orientation and manifold connectivity.
    """
    corners = topology.incidence[2, 0] * vertices                               # [3] Point-[F]
    planes = (corners[0] & corners[1] & corners[2]).normalized()                # Plane-[F]
    face_quadrics = planes * (planes & Point)                                   # (Plane+ <- Point-)[F]
    # Sum incident face quadrics onto each vertex via the adjugate incidence map.
    quadrics = topology.incidence[2, 0].sum(axis=0).adjugate() * face_quadrics  # (Plane+ <- Point-)[V]
    yield topology, vertices

    for target in targets:
        V, E, F = topology.n_elements
        while F > target:
            ends = topology.incidence[1, 0]                                     # [2] Scalar+[E <- V]
            edge_quadrics = ends.sum(axis=0) * quadrics                         # (Plane+ <- Point-)[E]
            midpoints = ends.mean(axis=0) * vertices                            # Point-[E]
            # Solve each edge's 3x3 least-squares problem independently across the batch of edges.
            optima = collapse_position(edge_quadrics.batch(), midpoints.batch()).field()    # Point-[E]
            errors = edge_quadrics(optima) & optima                             # Scalar+[E]

            # Test every non-degenerate triangle moved by an edge contraction to ensure its plane keeps its orientation and nonzero area.
            corners = topology.incidence[2, 0] * vertices                       # [3] Point-[F]
            tail, head = ends * vertices                                        # Point-[E], Point-[E]
            # Each corner's opposite edge oriented along the triangle's boundary; joining it with the trial vertex forms the moved triangle's plane.
            facing_sides = topology.boundary[2, 1] * (tail & head)              # [3] Line-[F]
            at_edges, at_corners = plumbing.trials(topology)
            old_planes = at_corners.sum(axis=0) * (corners[0] & corners[1] & corners[2])    # Plane-[n_trials]
            new_planes = (at_corners * facing_sides).sum(axis=0) & (at_edges * optima)      # Plane-[n_trials]
            valid_trials = (old_planes.scalar_product(new_planes) > 0) & (new_planes.squared() > old_planes.squared() * 1e-12)
            legal = at_edges.adjugate()(~valid_trials) <= 0                     # [E]
            chosen_edges = plumbing.select_edges(topology, errors, legal, max(1, (F - target) // 2))

            # Selected vertex stars share no moved triangles, so their contractions apply simultaneously without invalidating the trial checks.
            topology, kept, merged = plumbing.contract(topology, chosen_edges)
            vertices = kept * vertices + merged * optima                        # Point-[V_coarser]
            quadrics = kept * quadrics + merged * edge_quadrics                 # (Plane+ <- Point-)[V_coarser]
            V, E, F = topology.n_elements
        yield topology, vertices

In [ ]:
surface_size = 2
face_targets = np.array([3500, 1800, 900, 450, 250])
duration_ms = 600
bunny_topology, bunny_coordinates = shapes.surface(MODEL_DIR / "bunny.obj", surface_size)
# Rotate the model's vertical axis from y to z.
upright = (mv.yz * (-np.pi / 4)).exp()                                          # [] Rotor+
bunny_vertices = upright >> mv.point(bunny_coordinates).field()                 # Point-[V]

bunny_states = show_states(simplify(bunny_topology, bunny_vertices, face_targets),
                           render.animate_surface, bunny_vertices,
                           name="qem_bunny", duration_ms=duration_ms,
                           n_frames=len(face_targets) + 1, stepped=True)

Flat shading from a fixed camera shows smaller triangles concentrating along the high-curvature ears, muzzle, and silhouette while broad flanks merge into larger facets.

In [ ]:
# checks
# Each supporting plane passes through its triangle's three corners.
assert ((face_planes & corners).abs() < 1e-12).all()

# The optimal replacement vertex zeroes the spatial polar plane, stays on the y == 0 ridge toward the steeper corner, and lowers joint error.
assert edge_quadric(optimum).norm() < 1e-9
assert (mv.y & optimum).abs() < 1e-12
assert 0 < (mv.x & optimum) < (mv.x & endpoints[0])
assert ((edge_quadric(optimum) & optimum) < (edge_quadric(endpoints) & endpoints)).all()

# Progressive contraction hits every target face count while keeping the surface closed of genus zero.
V_bunny, E_bunny, F_bunny = bunny_topology.n_elements
assert [len(step_topology.elements[2]) for step_topology, _ in bunny_states] == [F_bunny, *face_targets]
coarsest_topology, _ = bunny_states[-1]
assert coarsest_topology.is_closed and coarsest_topology.euler_characteristic == 2